# Demo: Dataset Middleware

An agent builds an RDF graph through middleware-provided tools, and we inspect the run with live tracing.

---

- **What is `DatasetMiddleware`?** A set of tools, prompts, and supporting infrastructure for LangChain agents.
- **What problem does it solve for a Research Agent?** Frontier models are improving their logical reasoning and mathematical skills all the time.
   - A structured memory is a formal representation of facts, which can **reduce cognitive burden**; e.g., fewer context tokens and fewer agent reasoning cycles.
   - Agents are encouraged to express facts in an _atemporal_ description logic; this avoids the problem of **stale facts accumulating in agent history**. Instead, a bounded description about some facts would automatically describe temporal context and any other necessary metadata.
- **What will this notebook demonstrate?** How to get an agent converting unstructured text into structured knowledge as quickly as possible. Also, how to trace the agent's activity while it does so.

## 1. Setup

We've got a few things to do before the demo gets started.

First, we have to filter a langchain-internal warning (unrelated to our data model, it's just annoying).

In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
    category=UserWarning,
)

### 1.1. Bring your own Model

This notebook uses [LangChain's OpenAI integration](https://docs.langchain.com/oss/python/integrations/chat/openai) for [a self-hosted model provider](https://docs.langchain.com/langsmith/playground-model-providers#openai-compatible-endpoint), but you can use any of [LangChain's supported providers](https://docs.langchain.com/oss/python/integrations/providers/all_providers)
What's important is that Deepagents requires a chat model capable of tool invocations.

_NOTE: The only reason for `streaming=False` is that the Ministral model that I'm using starts to embed tool invocations in its normal chat output, causing us to lose tool calls.
Your mileage may vary depending on the model you select._

In [2]:
import os

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    seed=42,
    temperature=1,
)

## 2. The Demonstration

### 2.1. Prompt Setup

The main deviation from the `SYSTEM_PROMPT` of the [baseline demonstration](./demo-baseline-ontology-extraction.ipynb) is that we no longer include `DATASET_TIPS`.

Recall that `DATASET_TIPS` was **tool-agnostic guidance extracted from the DatasetMiddleware's prompt**. In our baseline demonstration, it was used to reduce prompt asymmetry.

In [3]:
from typing import Final

from demo_utils import CORE_PROMPT, STOPPING_CRITERIA, VOCABULARY_TIPS

# Reduce prompt asymmetry without recreating middleware-only capabilities.
SYSTEM_PROMPT: Final[str] = CORE_PROMPT + VOCABULARY_TIPS + STOPPING_CRITERIA

### 2.2. Agent Task Definition

Here we finally present the agent with a problem: Extract an ontology from this data.

- We are deliberately vague, simply asking if we can formalize the data
   - The given persona tells the agent to focus on the semantic modeling problems that are a throughline of the text.
- We add extraneous information, which we shouldn't see in the output.
- Because this demonstration doesn't use `RDFVocabularyMiddleware`, the agent doesn't have the ability to find established vocabularies. Instead it will just make up terms. After all, we're asking it to assert **ALL** facts that are claimed by the content.
   - It's not enough to ask an agent to be grounded; you have to afford it references that it can use to ground itself.

In [4]:
from demo_utils import TASK

print(TASK)


Please assist me in representing the subject matter of the following text as an RDF graph.
Please use <urn:ex:> as the base for any IRIs that you need to mint as part
of your response. Be sure to label any new terms or properties that you mint
so that they are human readable.

```text
John is a person. Modern people are classified as _homo sapiens_. Apparently,
homo sapiens falls under the subtribe of Hominina. Every Hominidae is a
Haplorhini, and now the names don't even sound like they're related. Then
we get to primates, which can be controversial for some people, for some reason.
Nobody argues with the idea that primates are mammals, yet some people take
umbrage with the idea that _homo sapiens_ is an animal. Oh, Hominidae contains
Hominina, too. Emotions and theological views aside, can we formalize this?
```



#### 2.2.1 Desired Output

The agent doesn't have much (any) background knowledge, so we're not going to penalize it for missing best-practices:

1. Concepts that this document introduces sould be documented:
    1.1. They SHOULD have a [`rdfs:label`](https://www.w3.org/TR/rdf-schema/#ch_label)
    1.2. They SHOULD have a [`skos:definition`](https://www.w3.org/TR/skos-reference/#notes)
    1.2. They SHOULD have a description in the form of [`dcterms:description`](https://www.dublincore.org/specifications/dublin-core/dcmi-terms/#description), or at least a [`rdfs:comment`](https://www.w3.org/TR/rdf-schema/#ch_comment)

In [5]:
from demo_utils import expected_graph
from rdflib import Graph

print(expected_graph.serialize(format="turtle"))

@prefix ex: <urn:ex:> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Haplorhini a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Primate .

ex:Hominidae a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Haplorhini .

ex:Hominina a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Hominidae .

ex:HomoSapiens a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Hominina .

ex:Mammal a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Animal .

ex:Primate a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Mammal .

ex:John a ex:Person .

ex:Person rdfs:subClassOf ex:HomoSapiens ;
    owl:equivalentClass ex:HomoSapiens .




### 2.3. Tracing the Agent

**TODO:** Acknowledge that we extend the `recursion_limit` for the model & how much longer it takes the model to generate a result.

#### 2.3.1. Traces for Humans



In [6]:
from deepagents import create_deep_agent
from langgraph.errors import GraphRecursionError
from rdflib import Namespace
from rdflib_reasoning.middleware.dataset_middleware import (
    DatasetMiddleware,
    DatasetMiddlewareConfig,
)
from rdflib_reasoning.middleware.ministral_middleware import (
    MinistralPromptSuffixMiddleware,
)
from rdflib_reasoning.middleware.namespaces.spec_whitelist import (
    RestrictedNamespaceWhitelist,
)
from rdflib_reasoning.middleware.tracing_notebook import LiveNotebookTrace

dataset_middleware = DatasetMiddleware(
    config=DatasetMiddlewareConfig(
        namespace_whitelist=RestrictedNamespaceWhitelist().plus_entries(
            ("ex", Namespace("urn:ex:"))
        )
    )
)

with LiveNotebookTrace(heading="Dataset Trace") as trace:
    agent = trace.attach(
        create_deep_agent(
            model=llm,
            system_prompt=SYSTEM_PROMPT,
            middleware=[
                dataset_middleware,
                MinistralPromptSuffixMiddleware(),
            ],
        )
    )

    RUN_INPUT = {"messages": [{"role": "user", "content": TASK}]}
    # RUN_CONFIG = {"recursion_limit": 60}
    RUN_CONFIG = {}
    try:
        agent_response = agent.invoke(RUN_INPUT, config=RUN_CONFIG)
    except GraphRecursionError:
        display(trace.sink.snapshot())
        raise

# Dataset Trace

## Turn 1

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Person"
    },
    {
      "object": "Person",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Person"
    },
    {
      "object": "A biological classification for humans.",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Person"
    }
  ]
}
```

#### Rejection

```json
{
  "content": "Error invoking tool 'add_triples' with kwargs {'triples': [{'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'object': 'http://www.w3.org/2000/01/rdf-schema#Class'}, {'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#label', 'object': 'Person'}, {'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#comment', 'object': 'A biological classification for humans.'}]} with error:\n triples.2.object: Value error, Could not parse RDF term from A biological classification for humans.. If this is plain text, encode it as an RDF literal like \"\\\"A biological classification for humans.\\\"\". If this is an IRI, provide it either in canonical N3 form like <urn:example:Thing> or as a bare RFC 3987 IRI.\n Please fix the error and try again.",
  "status": "error",
  "tool_call_id": "M35IihxPo"
}
```

## Turn 2

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Person"
    },
    {
      "object": "\"Person\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Person"
    },
    {
      "object": "\"A biological classification for humans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Person"
    }
  ]
}
```

#### Result

```json
updated=3 message='3 of 3 triples added.'
```

## Turn 3

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "\"Homo Sapiens\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "\"The biological species classification for modern humans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "\"Hominina\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "\"A subtribe within the Hominini tribe, which includes modern humans and their direct ancestors.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "\"Hominidae\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "\"The family of great apes, which includes modern humans, chimpanzees, gorillas, and orangutans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Haplorhini"
    },
    {
      "object": "\"Haplorhini\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Haplorhini"
    },
    {
      "object": "\"The suborder of primates that includes humans, apes, and tarsiers.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Haplorhini"
    }
  ]
}
```

#### Result

```json
updated=12 message='12 of 12 triples added.'
```

## Turn 4

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "\"Primate\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "\"The order of mammals that includes humans, apes, monkeys, and prosimians.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "\"Mammal\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "\"A class of vertebrate animals that includes humans and other animals that nurse their young with milk and have hair or fur.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "\"Animal\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "\"A kingdom of living organisms that includes humans and other multi-cellular organisms characterized by movement and reproduction.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Property",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:subtribeOf"
    },
    {
      "object": "\"subtribe of\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:subtribeOf"
    },
    {
      "object": "\"Relates a taxonomic group to its subtribe.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:subtribeOf"
    }
  ]
}
```

#### Rejection

```json
{
  "content": "The term http://www.w3.org/2000/01/rdf-schema#Property is not in the vocabulary whitelist for this knowledge base. You SHOULD verify that you are using the correct term or consider the alternatives below. You MUST NOT call `add_triples` again with this same IRI.\n\nSuggested alternatives (Levenshtein distance measures string similarity; lower is better):\n\n| Qualified name | IRI | Levenshtein distance |\n| --- | --- | ---: |\n| `rdf:Property` | `http://www.w3.org/1999/02/22-rdf-syntax-ns#Property` | 1 |\n| `owl:onProperty` | `http://www.w3.org/2002/07/owl#onProperty` | 6 |\n\nIf none of the suggestions fit your intent, you MUST use a different term from the allowed vocabularies listed in the system prompt.",
  "status": "error",
  "tool_call_id": "VuuWG8DYt"
}
```

## Turn 5

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/1999/02/22-rdf-syntax-ns#Property",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:subtribeOf"
    },
    {
      "object": "\"subtribe of\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:subtribeOf"
    },
    {
      "object": "\"Relates a taxonomic group to its subtribe.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:subtribeOf"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "\"Primate\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "\"The order of mammals that includes humans, apes, monkeys, and prosimians.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "\"Mammal\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "\"A class of vertebrate animals that includes humans and other animals that nurse their young with milk and have hair or fur.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "\"Animal\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "\"A kingdom of living organisms that includes humans and other multi-cellular organisms characterized by movement and reproduction.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Animal"
    }
  ]
}
```

#### Result

```json
updated=3 message='3 of 12 triples added. 9 of 12 triples already existed.'
```

## Turn 6

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/1999/02/22-rdf-syntax-ns#Property",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:classifiedAs"
    },
    {
      "object": "\"classified as\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:classifiedAs"
    },
    {
      "object": "\"Relates a group or individual to its biological classification.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:classifiedAs"
    }
  ]
}
```

#### Result

```json
updated=3 message='3 of 3 triples added.'
```

## Turn 7

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "urn:ex:Person",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:John"
    },
    {
      "object": "John",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:John"
    },
    {
      "object": "urn:ex:Person",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:ModernPeople"
    },
    {
      "object": "urn:ex:HomoSapiens",
      "predicate": "urn:ex:classifiedAs",
      "subject": "urn:ex:ModernPeople"
    },
    {
      "object": "urn:ex:Hominina",
      "predicate": "urn:ex:subtribeOf",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "urn:ex:Haplorhini",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "urn:ex:Hominina",
      "predicate": "urn:ex:contains",
      "subject": "urn:ex:Hominidae"
    }
  ]
}
```

#### Result

```json
updated=7 message='7 of 7 triples added.'
```

## Turn 8

### Final Response

> I will define the property `ex:contains` with a label and comment and also represent the relationship between primates, mammals, and animals as specified in the text. To do this, I will:
> 1. Add a definition for `ex:contains`.
> 2. Represent `ex:Primate` as a subclass of `ex:Mammal`.
> 3. Represent `ex:Mammal` as a subclass of `ex:Animal`.
>
> Let's proceed with these additions.

### Tool: add_triples

#### Rejection

```json
{
  "content": "The term http://www.w3.org/2000/01/rdf-schema#Property is not in the vocabulary whitelist for this knowledge base. You SHOULD verify that you are using the correct term or consider the alternatives below. You MUST NOT call `add_triples` again with this same IRI.\n\nSuggested alternatives (Levenshtein distance measures string similarity; lower is better):\n\n| Qualified name | IRI | Levenshtein distance |\n| --- | --- | ---: |\n| `rdf:Property` | `http://www.w3.org/1999/02/22-rdf-syntax-ns#Property` | 1 |\n| `owl:onProperty` | `http://www.w3.org/2002/07/owl#onProperty` | 6 |\n\nIf none of the suggestions fit your intent, you MUST use a different term from the allowed vocabularies listed in the system prompt.",
  "status": "error",
  "tool_call_id": "VuuWG8DYt"
}
```

### Tool: add_triples

#### Rejection

```json
{
  "content": "Error invoking tool 'add_triples' with kwargs {'triples': [{'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'object': 'http://www.w3.org/2000/01/rdf-schema#Class'}, {'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#label', 'object': 'Person'}, {'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#comment', 'object': 'A biological classification for humans.'}]} with error:\n triples.2.object: Value error, Could not parse RDF term from A biological classification for humans.. If this is plain text, encode it as an RDF literal like \"\\\"A biological classification for humans.\\\"\". If this is an IRI, provide it either in canonical N3 form like <urn:example:Thing> or as a bare RFC 3987 IRI.\n Please fix the error and try again.",
  "status": "error",
  "tool_call_id": "M35IihxPo"
}
```


### 2.4. Manually Assessing Results

### 2.5. Accessing the Knowledge Base

You may have noticed that we retained a reference to the middleware when creating the agent:

```python
dataset_middleware = DatasetMiddleware()
```

At any point (even during agent execution) you can take a snapshot of the current dataset.
This is a useful method to extract the structured knowledge base without asking the agent to understand RDF serialization formats.

In [7]:
from rdflib import Dataset

dataset: Dataset = dataset_middleware.session.snapshot_dataset()

print("There are", len(dataset), "triples in the dataset")
print("-" * 80)
print(dataset.serialize(format="turtle"))

There are 37 triples in the dataset
--------------------------------------------------------------------------------
@prefix ns1: <urn:ex:> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ns1:Animal a rdfs:Class ;
    rdfs:label "Animal" ;
    rdfs:comment "A kingdom of living organisms that includes humans and other multi-cellular organisms characterized by movement and reproduction." .

ns1:Haplorhini a rdfs:Class ;
    rdfs:label "Haplorhini" ;
    rdfs:comment "The suborder of primates that includes humans, apes, and tarsiers." .

ns1:Hominidae a rdfs:Class ;
    rdfs:label "Hominidae" ;
    rdfs:comment "The family of great apes, which includes modern humans, chimpanzees, gorillas, and orangutans." ;
    rdfs:subClassOf ns1:Haplorhini ;
    ns1:contains ns1:Hominina .

ns1:Hominina a rdfs:Class ;
    rdfs:label "Hominina" ;
    rdfs:comment "A subtribe within the Hominini tribe, which includes modern humans an

### 2.6. Metrics

We have an intersting set of problems in terms of assessing our agent's performance:

1. We are comparing sets (which constrains choices of metrics)
2. If a fact missing from our prediction set, it's _likely_ bad
3. If our prediction set contains additional facts, it's _not necessarily_ bad.

#### 2.6.1. Why Are Facts Missing?

If a fact is _missing_, then the agent probably worked around it.
It probably made up new prompt-relative RDF Vocabulary terms.

One normally needs to create at least _some_ domain-specific terms when applying semantic web technologies to novel areas, but it's a sledgehammer.
Each new term lacks shared semantics and requires consumers to learn and align with it.
Adding more application-specific terms makes it easier to express your data, but makes it less useful it is to other agents (human or otherwise).d

There's some alternative explanations for missing facts:

1. What if the agent didnt model things explicitly as classes, but as `skos:Concept`s?
Well, that would be _less_ correct, but it wouldn't be impossible to understand.

2. What if the agent uses a more relevent and published domain-specific vocabulary; one that it didn't make up?
In that respect, the agent would be _more_ correct than our labeled examples.
This isn't super likely, beacuse most agents don't have every RDF vocabulary memorized.

> **Requirement:** We need tooling for agents to find and explore new vocabularies.

#### 2.6.2. Extra Facts?

We aren't penalizing the agent for failing to adhere to best practices, but tha doesn't mean that they _won't_.
This may introduce additional documentation triples (`rdfs:label`, `skos:definition`, etc).

We may also get richer semantics.
[OWL Object Property Restrctions](https://www.w3.org/TR/2012/REC-owl2-syntax-20121211/#Object_Property_Restrictions)
could tell us about relationships that a new class is allowed to have, or certain relationships _exist_ even if they aren't
explicitly represented in the data.
[Shapes Constraint Language](https://www.w3.org/TR/shacl/) could tell us about the graph patterns that a concept is allowed to have, which tells us about graph neighborhoods that are, again, not necessarily in the data.

Agents can't do either of these things in a zero-shot scenario if they aren't aware that they are even things that they can do.

> **Requirement:** Agents needs affordances and the ability to build model patterns.

#### 2.6.3. Are Metrics Insurmountable?

The problems that I described above are those faced by agents with more sophisticated behavior.
An agent with little-to-no RDF knowledge attempting to solve this problem without any investigative tools has the deck stacked against them.
In my typical testing at this level of middleware, the agent has a significant tendency to invent vocabulary.
It's not often _wrong_, but it's **creating** a new and isolated ontology.

Given these constraints, _this_ particular model can be asssessed by 

In [8]:
from demo_utils import evaluate_actual_graph

actual_graph: Graph = dataset.default_graph
measures, graphs = evaluate_actual_graph(actual_graph)

intersection = graphs["intersection"]
missing = graphs["missing"]
extra = graphs["extra"]
union = graphs["union"]

In [9]:
import sys

if measures["counts"]["intersection"] > 0:
    print(intersection.serialize(format="turtle"))
else:
    print(
        "The agent's graph output did NOT intersect the expected graph.",
        file=sys.stderr,
    )

@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<urn:ex:Haplorhini> a rdfs:Class .

<urn:ex:Hominidae> a rdfs:Class ;
    rdfs:subClassOf <urn:ex:Haplorhini> .

<urn:ex:Hominina> a rdfs:Class .

<urn:ex:HomoSapiens> a rdfs:Class .

<urn:ex:Mammal> a rdfs:Class .

<urn:ex:Primate> a rdfs:Class .

<urn:ex:John> a <urn:ex:Person> .




In [10]:
print(missing.serialize(format="turtle"))

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<urn:ex:Haplorhini> a owl:Class ;
    rdfs:subClassOf <urn:ex:Primate> .

<urn:ex:Person> rdfs:subClassOf <urn:ex:HomoSapiens> ;
    owl:equivalentClass <urn:ex:HomoSapiens> .

<urn:ex:Hominidae> a owl:Class .

<urn:ex:Hominina> a owl:Class ;
    rdfs:subClassOf <urn:ex:Hominidae> .

<urn:ex:Mammal> a owl:Class ;
    rdfs:subClassOf <urn:ex:Animal> .

<urn:ex:Primate> a owl:Class ;
    rdfs:subClassOf <urn:ex:Mammal> .

<urn:ex:HomoSapiens> a owl:Class ;
    rdfs:subClassOf <urn:ex:Hominina> .




In [11]:
print(extra.serialize(format="turtle"))

@prefix ns1: <urn:ex:> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ns1:Animal a rdfs:Class .

ns1:Person a rdfs:Class .

ns1:Hominidae ns1:contains ns1:Hominina .

ns1:ModernPeople a ns1:Person ;
    ns1:classifiedAs ns1:HomoSapiens .

ns1:classifiedAs a rdf:Property .

ns1:subtribeOf a rdf:Property .

ns1:HomoSapiens ns1:subtribeOf ns1:Hominina .




In [12]:
from demo_utils import pprint_measures

pprint_measures(measures)

Expected triples: 21
Actual triples: 16
Intersection triples: 8
Missing triples: 13
Extra triples: 8
Union triples: 29
Precision: 0.500
Recall: 0.381
F1 score: 0.432
Jaccard Index: 0.276
Triple Edit Distance: 21
Normalized Triple Edit Distance: 0.724
Exact Match: False


## 3. Conclusions

> What did dataset middleware give us?

The middleware by itself simply gave the agent the ability to modify a knowledge base, and encouragement to do so when needed.
The prompt built into the middleware gives some general guidance as to how _facts_ should be represented, but it does not push the agent towards an understanding of _what the agent **should** model with those facts_.

> What comes next?

We'll allow an agent to explore standard vocabularies.
This will help to ground the agent's terminology.
Vocabularies describe how their terms should be used and what their use means.
This falls short of prescriptive guidance towards how the agent should model knowledge, but it should help the agent produce higher-quality results.